# 3D Gaussian Splatting — DJI Avata 360 Drone Footage

Reconstruct a photorealistic 3D scene from 360° drone video.

**Pre-computed:** COLMAP already ran locally (1 hour CPU) — 386/432 images registered, 50,958 3D points.

**This notebook:** Trains Gaussian Splatting on the COLMAP output using Colab A100.

**Upload to Drive before running:** `DroneCV/gaussian_splat/images_v2.zip` + `colmap_output.zip`

In [ ]:
import torch, os, subprocess, glob, zipfile, shutil
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
assert torch.cuda.is_available(), 'Need GPU!'

from google.colab import drive
drive.mount('/content/drive')
print('\n✅ Ready')

In [ ]:
# Load pre-computed COLMAP + images from Drive
DRIVE_DIR = '/content/drive/MyDrive/DroneCV/gaussian_splat_data'
WORK_DIR = '/content/gs_data'
os.makedirs(f'{WORK_DIR}/sparse/0', exist_ok=True)
os.makedirs(f'{WORK_DIR}/images', exist_ok=True)

# Extract images
print('Extracting images...')
img_zip = f'{DRIVE_DIR}/images_v3.zip'
with zipfile.ZipFile(img_zip, 'r') as z:
    for m in tqdm(z.namelist(), desc='Images'):
        z.extract(m, WORK_DIR)

# Extract COLMAP output
print('Extracting COLMAP output...')
colmap_zip = f'{DRIVE_DIR}/colmap_output.zip'
with zipfile.ZipFile(colmap_zip, 'r') as z:
    for m in tqdm(z.namelist(), desc='COLMAP'):
        z.extract(m, '/content/colmap_tmp')

# Move COLMAP files to expected structure
for f in glob.glob('/content/colmap_tmp/colmap/sparse/1/*'):
    shutil.copy(f, f'{WORK_DIR}/sparse/0/')

imgs = glob.glob(f'{WORK_DIR}/images/*.jpg')
print(f'\n✅ {len(imgs)} images + COLMAP sparse model ready')

In [ ]:
# Clone and install gaussian-splatting
%cd /content
!git clone https://github.com/graphdeco-inria/gaussian-splatting.git --recursive 2>&1 | tail -3
%cd gaussian-splatting
!pip install -q plyfile tqdm
!pip install submodules/diff-gaussian-rasterization 2>&1 | tail -2
!pip install submodules/simple-knn 2>&1 | tail -2
!pip install submodules/fused-ssim 2>&1 | tail -2
print('\n✅ Gaussian Splatting installed')

In [ ]:
# Train Gaussian Splatting (7000 iterations for quick results, 30000 for full quality)
%cd /content/gaussian-splatting

ITERATIONS = 7000  # Increase to 30000 for best quality
OUTPUT_DIR = '/content/gs_output'

!python train.py \
  -s {WORK_DIR} \
  --iterations {ITERATIONS} \
  --model_path {OUTPUT_DIR} \
  --sh_degree 3 \
  --test_iterations 1000 3000 7000

print(f'\n✅ Training complete! Output: {OUTPUT_DIR}')

In [ ]:
import glob

# Render novel views
!python render.py -s {WORK_DIR} --model_path {OUTPUT_DIR} --skip_test

# Show rendered results
renders = sorted(glob.glob(f"{OUTPUT_DIR}/train/ours_{ITERATIONS}/renders/*.png"))[:12]
if not renders:
    renders = sorted(glob.glob(f"{OUTPUT_DIR}/train/ours_*/renders/*.png"))[:12]
if renders:
    n = min(12, len(renders))
    rows = 2 if n > 6 else 1
    cols = min(6, n)
    fig, axes = plt.subplots(rows, cols, figsize=(3*cols, 3*rows))
    axes = np.array(axes).flatten()
    for i in range(n):
        img = plt.imread(renders[i])
        axes[i].imshow(img); axes[i].axis("off")
    plt.suptitle(f"Gaussian Splatting Renders ({ITERATIONS} iter)", fontsize=13)
    plt.tight_layout()
    plt.savefig("/content/gs_renders.png", dpi=150)
    plt.show()
    print(f"Rendered {len(renders)} views")
else:
    print("No renders — training may not have produced output images")
    print("But the model is saved at:", OUTPUT_DIR)


In [ ]:
# Save results to Drive
import shutil
SAVE_DIR = f'{DRIVE_DIR}/output'
os.makedirs(SAVE_DIR, exist_ok=True)

# Save the trained model
if os.path.exists(f'{OUTPUT_DIR}/point_cloud'):
    shutil.copytree(f'{OUTPUT_DIR}/point_cloud', f'{SAVE_DIR}/point_cloud', dirs_exist_ok=True)
    print(f'Saved point cloud to Drive')

# Save renders
if os.path.exists('/content/gs_renders.png'):
    shutil.copy('/content/gs_renders.png', f'{SAVE_DIR}/gs_renders.png')
    print('Saved renders image to Drive')

# Save a few individual renders
renders = sorted(glob.glob(f'{OUTPUT_DIR}/train/ours_{ITERATIONS}/renders/*.png'))[:6]
for r in renders:
    shutil.copy(r, f'{SAVE_DIR}/{os.path.basename(r)}')

print(f'\n✅ All saved to {SAVE_DIR}')

## Summary

| Step | Where | Result |
|------|-------|--------|
| Perspective extraction | Local | 432 images from 8K equirectangular |
| COLMAP SfM | Local (1h CPU) | 386/432 registered, 50,958 3D points |
| Gaussian Splatting | Colab A100 | Photorealistic 3D reconstruction |

**Next:** Use trained model for GNSS-denied localization, site inspection, change detection.